In [2]:
#Install pyspark for use of the examples
pip install pyspark

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 434.2 MB 30 kB/s s eta 0:00:01
     |████████████████████████████████| 203 kB 8.5 MB/s eta 0:00:01
  Created wheel for pyspark: filename=pyspark-4.0.2-py2.py3-none-any.whl size=434827108 sha256=63d8cd3e2e074d0affef6ec18c98792e66692ec9d823e5f5e1e0b1fc01f9b07a
  Stored in directory: /Users/bradypinter/Library/Caches/pip/wheels/43/2f/d1/97fb12783fd5d209434848fbd4b897874d7bb2679e7dfc0db3
Successfully built pyspark
You should consider upgrading via the '/Applications/Xcode.app/Contents/Developer/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [1]:
#Import needed libraries for use
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, avg, count, sum as spark_sum,
    when, desc, round as spark_round
)

In [2]:
#Set up our Spark Session for use.
spark = (
    SparkSession.builder
    .appName("PySpark Intro")    
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/09 21:35:50 WARN Utils: Your hostname, Bradys-MacBook-Pro-2.local, resolves to a loopback address: 127.0.0.1; using 172.21.33.106 instead (on interface en0)
26/03/09 21:35:50 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/09 21:35:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
#Creation of our dataset
data = [
    (1,  "Alice",   "Electronics", "Laptop",      1200.00, 1, "2024-01-15"),
    (2,  "Bob",     "Clothing",    "Jacket",         89.99, 2, "2024-01-17"),
    (3,  "Alice",   "Electronics", "Headphones",    199.99, 1, "2024-01-20"),
    (4,  "Charlie", "Books",       "Python Guide",   34.99, 3, "2024-02-01"),
    (5,  "Bob",     "Electronics", "Tablet",        450.00, 1, "2024-02-05"),
    (6,  "Diana",   "Clothing",    "Sneakers",      120.00, 1, "2024-02-10"),
    (7,  "Charlie", "Electronics", "Smartwatch",    299.99, 2, "2024-02-14"),
    (8,  "Diana",   "Books",       "Data Science",   49.99, 1, "2024-03-01"),
    (9,  "Alice",   "Clothing",    "T-Shirt",        25.00, 4, "2024-03-05"),
    (10, "Bob",     "Books",       "Spark Guide",    39.99, 2, "2024-03-10"),
]


columns = ["order_id", "customer", "category", "product",
           "unit_price", "quantity", "order_date"]

df = spark.createDataFrame(data, schema=columns)

# Add a derived column: total_price = unit_price * quantity
df = df.withColumn("total_price", spark_round(col("unit_price") * col("quantity"), 2))

print("\n" + "="*60)
print("  SAMPLE DATASET — E-Commerce Orders")
print("="*60)
df.show(truncate=False)
df.printSchema()


  SAMPLE DATASET — E-Commerce Orders


+--------+--------+-----------+------------+----------+--------+----------+-----------+
|order_id|customer|category   |product     |unit_price|quantity|order_date|total_price|
+--------+--------+-----------+------------+----------+--------+----------+-----------+
|1       |Alice   |Electronics|Laptop      |1200.0    |1       |2024-01-15|1200.0     |
|2       |Bob     |Clothing   |Jacket      |89.99     |2       |2024-01-17|179.98     |
|3       |Alice   |Electronics|Headphones  |199.99    |1       |2024-01-20|199.99     |
|4       |Charlie |Books      |Python Guide|34.99     |3       |2024-02-01|104.97     |
|5       |Bob     |Electronics|Tablet      |450.0     |1       |2024-02-05|450.0      |
|6       |Diana   |Clothing   |Sneakers    |120.0     |1       |2024-02-10|120.0      |
|7       |Charlie |Electronics|Smartwatch  |299.99    |2       |2024-02-14|599.98     |
|8       |Diana   |Books      |Data Science|49.99     |1       |2024-03-01|49.99      |
|9       |Alice   |Clothing   |T

**Question 1 - Which customer had the largest total_price?**

In [6]:
answer1 = df.orderBy(desc("total_price"))
answer1.show(truncate=False)

+--------+--------+-----------+------------+----------+--------+----------+-----------+
|order_id|customer|category   |product     |unit_price|quantity|order_date|total_price|
+--------+--------+-----------+------------+----------+--------+----------+-----------+
|1       |Alice   |Electronics|Laptop      |1200.0    |1       |2024-01-15|1200.0     |
|7       |Charlie |Electronics|Smartwatch  |299.99    |2       |2024-02-14|599.98     |
|5       |Bob     |Electronics|Tablet      |450.0     |1       |2024-02-05|450.0      |
|3       |Alice   |Electronics|Headphones  |199.99    |1       |2024-01-20|199.99     |
|2       |Bob     |Clothing   |Jacket      |89.99     |2       |2024-01-17|179.98     |
|6       |Diana   |Clothing   |Sneakers    |120.0     |1       |2024-02-10|120.0      |
|4       |Charlie |Books      |Python Guide|34.99     |3       |2024-02-01|104.97     |
|9       |Alice   |Clothing   |T-Shirt     |25.0      |4       |2024-03-05|100.0      |
|10      |Bob     |Books      |S